# OCR Stiker Obat - End to End
Pipeline: **YOLO (cari `entete`) -> Orientation (0/90/180/270) -> OCR (ResNet18 + Transformer + CTC)**

Urutan jalan (atas ke bawah):
| Bagian | Isi | Catatan |
|---|---|---|
| 0 | Setup & fungsi bantu | selalu jalan |
| 1 | Siapkan data (YOLO + crop) | sekali saja |
| 2 | **Labeling manual** | interaktif, perlu >= ~200 crop |
| 3 | Train YOLO | |
| 4 | Train Orientation | butuh label dari bagian 2 |
| 5 | Train OCR | butuh label dari bagian 2 |
| 6 | Inference -> `submission.csv` | satu-satunya bagian yang dipakai saat submit |

Struktur folder: `data/train`, `data/test`, `data/sample_submission.csv`. Folder `work/` dan `models/` dibuat otomatis.

## 0. Setup

In [6]:
# pip install ultralytics rapidfuzz     # jalankan sekali kalau belum ada
import json, random, re, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA, WORK, MODELS = Path("data"), Path("work"), Path("models")
CROPS, CSV = WORK / "crops", WORK / "labels.csv"
for d in [WORK / "yolo/images/train", WORK / "yolo/images/val",
          WORK / "yolo/labels/train", WORK / "yolo/labels/val", CROPS, MODELS]:
    d.mkdir(parents=True, exist_ok=True)

SEED       = 42
ENTETE_ID  = 1        # COCO: 0='Images' (dummy), 1='entete', 2='vin'
PAD        = 4        # padding crop (px)
IMGSZ      = 416      # ukuran input YOLO
CONF       = 0.25     # ambang confidence YOLO
OCR_H, OCR_W = 96, 384   # ukuran input OCR (boleh di-tuning lewat CER validasi)
ORI_SIZE   = 128         # ukuran input orientation
PRETRAINED = True        # bobot ImageNet butuh internet SEKALI saat training; set False kalau offline

# Vocab OCR (aturan submission: A-Z, 0-9, '/', '-').  Index 0 = blank CTC
CHARS = "-/0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
char_to_idx = {c: i + 1 for i, c in enumerate(CHARS)}
idx_to_char = {i + 1: c for i, c in enumerate(CHARS)}
NUM_CLASSES = len(CHARS) + 1

random.seed(SEED); np.random.seed(SEED)

In [7]:
# ---------- Fungsi bantu (tanpa torch) ----------
SKIP = "__SKIP__"      # penanda crop yang dilewati saat labeling

def clean_text(text):
    """Huruf besar, buang semua selain A-Z 0-9 / -  (spasi, ':', '.', dll ikut dibuang)."""
    return re.sub(r"[^A-Z0-9/\-]", "", str(text).upper())

# KONVENSI ROTASI: rot = derajat putar SEARAH JARUM JAM agar teks tegak. Kelas orientation = rot // 90.
_CV_ROT = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}

def rotate_cw(img, rot):
    return img if rot % 360 == 0 else cv2.rotate(img, _CV_ROT[rot % 360])

def needed_rot(applied):
    """Gambar tegak yang diputar `applied` derajat cw butuh berapa derajat cw untuk ditegakkan lagi."""
    return (360 - applied) % 360

def augment(img):
    """Augmentasi ringan (BGR uint8). Dipakai hanya saat training."""
    h, w = img.shape[:2]
    if random.random() < 0.7:                                   # kecerahan & kontras
        img = cv2.convertScaleAbs(img, alpha=random.uniform(0.7, 1.3), beta=random.uniform(-30, 30))
    if random.random() < 0.3:                                   # blur
        k = random.choice([3, 5]); img = cv2.GaussianBlur(img, (k, k), 0)
    if random.random() < 0.3:                                   # miring/zoom tipis
        M = cv2.getRotationMatrix2D((w / 2, h / 2), random.uniform(-2, 2), random.uniform(0.95, 1.05))
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    if random.random() < 0.2:                                   # noise
        img = np.clip(img.astype(np.float32) + np.random.normal(0, 8, img.shape), 0, 255).astype(np.uint8)
    return img

def load_labeled():
    """Baris labels.csv yang sudah terisi (tanpa yang di-skip)."""
    df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
    df = df[(df["text"] != "") & (df["text"] != SKIP)].copy()
    df["rot"] = df["rot"].astype(int)
    return df

def load_upright(file, rot):
    """Crop yang sudah diputar sehingga teks tegak."""
    return rotate_cw(cv2.imread(str(CROPS / file)), rot)

def cer(preds, refs):
    """Character Error Rate = total jarak Levenshtein / total panjang referensi."""
    from rapidfuzz.distance import Levenshtein
    errors = sum(Levenshtein.distance(p, r) for p, r in zip(preds, refs))
    return errors / max(1, sum(len(r) for r in refs))

In [8]:
# ---------- Model (torch) ----------
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
print("device:", DEVICE)

def to_tensor(img_bgr, h, w):
    """Gambar BGR -> tensor (3, h, w) ternormalisasi. DIPAKAI SAMA di training & inference."""
    img = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), (w, h))
    img = (img.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(img).permute(2, 0, 1)

def build_orientation_model(pretrained=False):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, 4)                    # 4 kelas: 0/90/180/270
    return m

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))          # (max_len, 1, d_model)

    def forward(self, x):                                    # x: (T, B, d_model)
        return self.dropout(x + self.pe[: x.size(0)])

class OCRModel(nn.Module):
    """ResNet18 (lebar dijaga) -> tinggi di-pool jadi 1 -> Transformer Encoder -> CTC."""
    def __init__(self, num_classes=NUM_CLASSES, d_model=256, nhead=4, num_layers=2,
                 dim_ff=1024, dropout=0.1, pretrained=False):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        # Tinggi dikecilkan, LEBAR dijaga (lebar = sumbu waktu CTC).
        # conv1 DAN downsample harus diubah bersamaan, kalau tidak shape shortcut tidak cocok.
        for layer, stride in [(resnet.layer2, (2, 1)), (resnet.layer3, (1, 1)), (resnet.layer4, (1, 1))]:
            layer[0].conv1.stride = stride
            layer[0].downsample[0].stride = stride
        self.backbone = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                                      resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4)
        self.height_pool = nn.AdaptiveAvgPool2d((1, None))
        self.proj = nn.Conv2d(512, d_model, kernel_size=1)
        self.pos = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout), num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):                                    # x: (B, 3, H, W)
        x = self.proj(self.height_pool(self.backbone(x)))    # (B, d, 1, T)
        x = x.squeeze(2).permute(2, 0, 1)                    # (T, B, d)
        x = self.transformer(self.pos(x))
        return F.log_softmax(self.classifier(x), dim=2)      # (T, B, num_classes)

def ctc_decode(log_probs):
    """Greedy CTC decode. (T, B, C) -> list[str] sepanjang B."""
    best, texts = log_probs.argmax(2).cpu().numpy(), []
    for b in range(best.shape[1]):
        chars, prev = [], 0
        for k in best[:, b]:
            if k != 0 and k != prev:
                chars.append(idx_to_char[int(k)])
            prev = k
        texts.append("".join(chars))
    return texts

device: cuda


## 1. Siapkan data
Membuat (a) dataset YOLO satu kelas `entete`, (b) crop `entete` + `labels.csv`.
**Aman dijalankan ulang:** `labels.csv` tidak akan ditimpa kalau sudah ada (supaya label-mu tidak hilang).

In [11]:
coco = json.load(open(DATA / "train" / "_annotations.coco.json"))

# Perbaiki 1 annotation rusak (key '' dan 'catimage_idegory_id')
for ann in coco["annotations"]:
    if "image_id" not in ann:
        ann["image_id"], ann["category_id"] = ann.pop(""), ann.pop("catimage_idegory_id")

images = {img["id"]: img for img in coco["images"]}
entete_anns = [a for a in coco["annotations"] if a["category_id"] == ENTETE_ID]
print("gambar:", len(images), "| box entete:", len(entete_anns))

# Bagi train/val per GAMBAR (bukan per box) supaya tidak bocor
ids = sorted(images); random.Random(SEED).shuffle(ids)
val_ids = set(ids[: int(0.1 * len(ids))])

anns_by_img = {}
for a in entete_anns:
    anns_by_img.setdefault(a["image_id"], []).append(a)

# (a) dataset YOLO: kelas 0 = entete, koordinat dinormalisasi
for img_id, img in images.items():
    split = "val" if img_id in val_ids else "train"
    shutil.copy(DATA / "train" / img["file_name"], WORK / f"yolo/images/{split}" / img["file_name"])
    W, H = img["width"], img["height"]
    lines = [f"0 {(x + w/2)/W:.6f} {(y + h/2)/H:.6f} {w/W:.6f} {h/H:.6f}"
             for x, y, w, h in (a["bbox"] for a in anns_by_img.get(img_id, []))]
    (WORK / f"yolo/labels/{split}" / (Path(img["file_name"]).stem + ".txt")).write_text("\n".join(lines))

(WORK / "yolo/data.yaml").write_text(
    f"path: {(WORK / 'yolo').resolve()}\ntrain: images/train\nval: images/val\nnames:\n  0: entete\n")

# (b) crop entete + template labels.csv (HANYA kalau belum ada)
if CSV.exists():
    print("labels.csv sudah ada -> crop & label TIDAK ditimpa.")
else:
    rows = []
    for a in entete_anns:
        img = images[a["image_id"]]
        im = cv2.imread(str(DATA / "train" / img["file_name"]))
        x, y, w, h = [int(v) for v in a["bbox"]]
        crop = im[max(0, y - PAD): y + h + PAD, max(0, x - PAD): x + w + PAD]
        name = f"{Path(img['file_name']).stem}_{a['id']}.jpg"
        cv2.imwrite(str(CROPS / name), crop)
        rows.append({"file": name, "split": "val" if a["image_id"] in val_ids else "train", "rot": "", "text": ""})
    pd.DataFrame(rows).to_csv(CSV, index=False)
    print("crop tersimpan:", len(rows))

TypeError: string indices must be integers, not 'str'

## 2. Labeling manual
Satu jendela akan terbuka (Tkinter). Per crop:
1. **Ctrl+R** = putar 90 derajat searah jarum jam, sampai teks **tegak** dan terbaca normal  -> mengisi `rot`
2. Ketik isi teks dari baris atas ke bawah (spasi, `:`, `.` boleh, dibersihkan otomatis) lalu **Enter**
3. Tidak terbaca sama sekali -> **Lewati**

Tersimpan otomatis tiap Enter; boleh ditutup dan dibuka lagi (lanjut dari yang belum terlabel). Target awal **>= 200 crop**.

> **Cek dulu format label dari panitia** (apakah semua teks `LOT...FAB...PER...` digabung, atau hanya nilainya). Default di sini: semua teks yang terlihat, atas ke bawah.
>
> Kalau jendela tidak bisa dibuka (mis. Kaggle / Mac tertentu), labeling dilakukan di komputer lokal saja.

In [ ]:
def run_label_tool():
    import tkinter as tk
    from tkinter import messagebox
    from PIL import Image, ImageTk

    VIEW_W, VIEW_H = 900, 420
    df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
    order = list(df.index); random.Random(SEED).shuffle(order)      # acak supaya train & val sama-sama terisi
    start = next((p for p, i in enumerate(order) if df.at[i, "text"] == ""), 0)

    root = tk.Tk(); root.title("Label entete")
    state = {"pos": start, "rot": 0, "photo": None}

    canvas = tk.Label(root, bg="#222"); canvas.pack(padx=10, pady=10)
    info = tk.Label(root, font=("Arial", 11)); info.pack()
    preview = tk.Label(root, font=("Consolas", 12), fg="#0a6"); preview.pack()
    text_var = tk.StringVar()
    entry = tk.Entry(root, textvariable=text_var, font=("Consolas", 16), width=60); entry.pack(padx=10, pady=6)
    text_var.trace_add("write", lambda *_: preview.config(text="disimpan sebagai: " + clean_text(text_var.get())))

    def show():
        row = df.loc[order[state["pos"]]]
        im = Image.open(CROPS / row["file"]).convert("RGB").rotate(-state["rot"], expand=True)  # -rot = searah jarum jam
        s = min(VIEW_W / im.width, VIEW_H / im.height)
        im = im.resize((int(im.width * s), int(im.height * s)), Image.LANCZOS)
        state["photo"] = ImageTk.PhotoImage(im); canvas.config(image=state["photo"])
        done = int((df["text"] != "").sum())
        info.config(text=f"[{state['pos'] + 1}/{len(order)}]  {row['file']}  |  rot = {state['rot']}  |  terlabel: {done}")

    def goto(pos):
        state["pos"] = max(0, min(len(order) - 1, pos))
        row = df.loc[order[state["pos"]]]
        state["rot"] = int(row["rot"]) if row["rot"] != "" else 0
        text_var.set("" if row["text"] in ("", SKIP) else row["text"])
        show(); entry.focus_set()

    def rotate(_=None):
        state["rot"] = (state["rot"] + 90) % 360; show()

    def save_and_next(_=None):
        cleaned = clean_text(text_var.get())
        if not cleaned:
            messagebox.showwarning("Teks kosong", "Isi teksnya dulu, atau pakai Lewati."); return
        i = order[state["pos"]]
        df.at[i, "rot"], df.at[i, "text"] = str(state["rot"]), cleaned
        df.to_csv(CSV, index=False); goto(state["pos"] + 1)

    def skip(_=None):
        i = order[state["pos"]]
        df.at[i, "rot"], df.at[i, "text"] = "0", SKIP
        df.to_csv(CSV, index=False); goto(state["pos"] + 1)

    bar = tk.Frame(root); bar.pack(pady=6)
    for label, fn in [("Kembali", lambda: goto(state["pos"] - 1)), ("Putar 90 (Ctrl+R)", rotate),
                      ("Lewati", skip), ("Simpan & Lanjut (Enter)", save_and_next)]:
        tk.Button(bar, text=label, command=fn, width=22).pack(side="left", padx=4)
    root.bind("<Return>", save_and_next); root.bind("<Control-r>", rotate)
    goto(state["pos"]); root.mainloop()

run_label_tool()

In [ ]:
 # Cek progres labeling
labeled = load_labeled()
print("terlabel:", len(labeled), "| train:", (labeled.split == "train").sum(), "| val:", (labeled.split == "val").sum())
print("rot:", labeled["rot"].value_counts().to_dict())
labeled[["file", "rot", "text"]].head(8)

## 3. Train YOLO
Satu kelas (`entete`), kotak biasa. Butuh `yolov8s.pt` (terunduh otomatis sekali; kalau offline, taruh filenya di folder ini).
Tidak perlu menunggu labeling teks, bagian ini bisa dijalankan kapan saja setelah bagian 1.

In [ ]:
from ultralytics import YOLO

yolo_train = YOLO("yolov8s.pt")
yolo_train.train(
    data=str(WORK / "yolo/data.yaml"), imgsz=IMGSZ, epochs=60, patience=15, batch=16,
    project=str(WORK / "runs"), name="entete", seed=SEED,
    hsv_v=0.5, hsv_s=0.5,      # test lebih gelap & lebih tajam (dari EDA) -> augmentasi warna lebih kuat
)
shutil.copy(Path(yolo_train.trainer.best), MODELS / "yolo_entete.pt")
m = YOLO(MODELS / "yolo_entete.pt").val(data=str(WORK / "yolo/data.yaml"), imgsz=IMGSZ)
print("mAP50:", round(m.box.map50, 4), "| mAP50-95:", round(m.box.map, 4))

In [ ]:
# Cek detektor di TEST (tidak punya label): berapa gambar terdeteksi & seberapa yakin
yolo = YOLO(MODELS / "yolo_entete.pt")
test_paths = sorted(p for p in (DATA / "test").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
confs = [float(r.boxes.conf.max()) for r in yolo.predict([str(p) for p in test_paths], imgsz=IMGSZ,
                                                         conf=CONF, stream=True, verbose=False) if len(r.boxes)]
print(f"test: {len(test_paths)} gambar | terdeteksi: {len(confs)} ({len(confs)/len(test_paths):.1%})"
      f" | conf median: {np.median(confs):.3f} | min: {min(confs):.3f}")

## 4. Train Orientation (4 kelas)
Dari tiap crop berlabel (yang sudah ditegakkan), dibuat 4 versi putaran; modelnya belajar menebak putaran koreksinya.

In [ ]:
class OriDataset(Dataset):
    def __init__(self, df, train):
        self.items, self.train = list(zip(df["file"], df["rot"])), train
    def __len__(self):
        return len(self.items) * (8 if self.train else 4)     # train: 8x per epoch (augmentasi acak)
    def __getitem__(self, i):
        if self.train:
            (file, rot), applied = self.items[i % len(self.items)], random.choice([0, 90, 180, 270])
        else:
            (file, rot), applied = self.items[i // 4], (i % 4) * 90
        img = rotate_cw(load_upright(file, rot), applied)
        if self.train:
            img = augment(img)
        return to_tensor(img, ORI_SIZE, ORI_SIZE), needed_rot(applied) // 90

lab = load_labeled()
assert len(lab) >= 50, f"Baru {len(lab)} crop berlabel - lengkapi dulu di bagian 2."
assert (lab.split == "val").sum() > 0, "Belum ada crop val yang berlabel - lanjutkan labeling (urutannya sudah diacak)."
ori_train = DataLoader(OriDataset(lab[lab.split == "train"], True), batch_size=32, shuffle=True)
ori_val   = DataLoader(OriDataset(lab[lab.split == "val"], False), batch_size=32)

ori_model = build_orientation_model(pretrained=PRETRAINED).to(DEVICE)
opt = torch.optim.AdamW(ori_model.parameters(), lr=3e-4, weight_decay=1e-4)
best_acc = 0.0
for epoch in range(12):
    ori_model.train()
    for x, y in tqdm(ori_train, desc=f"ori epoch {epoch+1}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); F.cross_entropy(ori_model(x), y).backward(); opt.step()

    ori_model.eval(); hit = tot = 0
    with torch.no_grad():
        for x, y in ori_val:
            hit += (ori_model(x.to(DEVICE)).argmax(1).cpu() == y).sum().item(); tot += len(y)
    acc = hit / tot
    print(f"epoch {epoch+1:2d} | val acc {acc:.3f}")
    if acc >= best_acc:
        best_acc = acc; torch.save(ori_model.state_dict(), MODELS / "orientation.pth")
print("terbaik:", round(best_acc, 3), "-> models/orientation.pth")

## 5. Train OCR
Metrik: **CER** di crop validasi (makin kecil makin bagus). Data berlabel masih sedikit, jadi tiap epoch mengulang data 10x dengan augmentasi acak.

In [ ]:
class OCRDataset(Dataset):
    def __init__(self, df, train, repeat=10):
        self.items, self.train, self.repeat = list(zip(df["file"], df["rot"], df["text"])), train, repeat
    def __len__(self):
        return len(self.items) * (self.repeat if self.train else 1)
    def __getitem__(self, i):
        file, rot, text = self.items[i % len(self.items)]
        img = load_upright(file, rot)
        if self.train:
            img = augment(img)
        target = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
        return to_tensor(img, OCR_H, OCR_W), target

def collate(batch):
    xs, ys = zip(*batch)
    lens = torch.tensor([len(y) for y in ys])
    padded = torch.zeros(len(ys), int(lens.max()), dtype=torch.long)
    for i, y in enumerate(ys):
        padded[i, : len(y)] = y
    return torch.stack(xs), padded, lens

lab = load_labeled()
assert len(lab) >= 50, f"Baru {len(lab)} crop berlabel - lengkapi dulu di bagian 2."
assert (lab.split == "val").sum() > 0, "Belum ada crop val yang berlabel - lanjutkan labeling (urutannya sudah diacak)."
tr, va = lab[lab.split == "train"], lab[lab.split == "val"]
ocr_train = DataLoader(OCRDataset(tr, True), batch_size=16, shuffle=True, collate_fn=collate)
ocr_val   = DataLoader(OCRDataset(va, False), batch_size=16, collate_fn=collate)

ocr_model = OCRModel(pretrained=PRETRAINED).to(DEVICE)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
opt = torch.optim.AdamW(ocr_model.parameters(), lr=3e-4, weight_decay=0.01)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=4)

best_cer, bad_epochs = 1e9, 0
for epoch in range(80):
    ocr_model.train(); losses = []
    for x, y, lens in tqdm(ocr_train, desc=f"ocr epoch {epoch+1}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = ocr_model(x)                                               # (T, B, C)
        in_lens = torch.full((x.size(0),), out.size(0), dtype=torch.long)
        loss = ctc(out, y, in_lens, lens)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(ocr_model.parameters(), 5.0)            # cegah gradient meledak
        opt.step(); losses.append(loss.item())

    ocr_model.eval(); preds, refs = [], []
    with torch.no_grad():
        for x, y, lens in ocr_val:
            preds += ctc_decode(ocr_model(x.to(DEVICE)))
            refs  += ["".join(idx_to_char[int(k)] for k in y[i, : lens[i]]) for i in range(len(lens))]
    val_cer = cer(preds, refs); sched.step(val_cer)
    print(f"epoch {epoch+1:2d} | loss {np.mean(losses):.3f} | val CER {val_cer:.4f}")

    if val_cer < best_cer:
        best_cer, bad_epochs = val_cer, 0
        torch.save({"model": ocr_model.state_dict(), "ocr_h": OCR_H, "ocr_w": OCR_W}, MODELS / "ocr.pth")
    else:
        bad_epochs += 1
        if bad_epochs >= 10:
            print("early stopping"); break
print("CER terbaik:", round(best_cer, 4), "-> models/ocr.pth")

In [ ]:
# Contoh prediksi di crop validasi (cek dengan mata, bukan cuma angka CER)
for p, r in list(zip(preds, refs))[:10]:
    print("OK " if p == r else "XX ", "benar:", r, "\n    tebak:", p)

## 6. Inference -> `submission.csv`
Bagian ini **mandiri**: hanya butuh 3 file di `models/` dan fungsi-fungsi dari bagian 0.
Untuk Kaggle: salin bagian 0 + bagian 6 ke notebook baru, upload `models/` sebagai dataset, lalu ubah `MODELS`, `TEST_DIR`, `SAMPLE_SUB`. Set `PRETRAINED = False` (internet mati).

Kalau YOLO tidak menemukan apa-apa, teks diisi kosong (CER = 1.0 untuk gambar itu, tidak lebih buruk dari menebak asal).

In [ ]:
TEST_DIR   = DATA / "test"
SAMPLE_SUB = DATA / "sample_submission.csv"

yolo = YOLO(MODELS / "yolo_entete.pt")

ori = build_orientation_model(pretrained=False)
ori.load_state_dict(torch.load(MODELS / "orientation.pth", map_location=DEVICE))
ori = ori.to(DEVICE).eval()

ocr = OCRModel(pretrained=False)
ocr.load_state_dict(torch.load(MODELS / "ocr.pth", map_location=DEVICE)["model"])
ocr = ocr.to(DEVICE).eval()

@torch.no_grad()
def read_image(path):
    if not path.exists():
        return ""
    r = yolo.predict(source=str(path), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    if len(r.boxes) == 0:
        return ""
    best = int(r.boxes.conf.argmax())                               # ambil deteksi paling yakin
    x1, y1, x2, y2 = [int(v) for v in r.boxes.xyxy[best].tolist()]
    crop = r.orig_img[max(0, y1 - PAD): y2 + PAD, max(0, x1 - PAD): x2 + PAD]
    if crop.size == 0:
        return ""
    k = ori(to_tensor(crop, ORI_SIZE, ORI_SIZE)[None].to(DEVICE)).argmax(1).item()   # 0..3
    crop = rotate_cw(crop, k * 90)
    return clean_text(ctc_decode(ocr(to_tensor(crop, OCR_H, OCR_W)[None].to(DEVICE)))[0])

sub = pd.read_csv(SAMPLE_SUB)
sub["text"] = [read_image(TEST_DIR / name) for name in tqdm(sub["id"])]
sub.to_csv("submission.csv", index=False)

assert sub["text"].str.fullmatch(r"[A-Z0-9/\-]*").all(), "ada karakter terlarang"
print("tersimpan: submission.csv |", len(sub), "baris | kosong:", int((sub["text"] == "").sum()))
sub.head()